<a href="https://colab.research.google.com/github/rehunter3502026-max/movi-reccomendation-CINEMATCH/blob/main/movie_reccomendation_netflix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import re
import string

In [ ]:
import pandas as pd
from google.colab import files


uploaded = files.upload()

# Read datasets
movie = pd.read_csv("/content/netflix_titles.csv.zip")
print("movie Dataset:")
display(movie.head())
movie.info()


Saving netflix_titles.csv.zip to netflix_titles.csv (4).zip
movie Dataset:


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer()
movie1=vectorizer.fit_transform(movie)
print(movie1)
from sklearn.feature_extraction.text import CountVectorizer
vectorizer=CountVectorizer()
movie2=vectorizer.fit_transform(movie)
print(movie2)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 12 stored elements and shape (12, 12)>
  Coords	Values
  (0, 9)	1.0
  (1, 11)	1.0
  (2, 10)	1.0
  (3, 4)	1.0
  (4, 0)	1.0
  (5, 1)	1.0
  (6, 2)	1.0
  (7, 8)	1.0
  (8, 7)	1.0
  (9, 5)	1.0
  (10, 6)	1.0
  (11, 3)	1.0
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 12 stored elements and shape (12, 12)>
  Coords	Values
  (0, 9)	1
  (1, 11)	1
  (2, 10)	1
  (3, 4)	1
  (4, 0)	1
  (5, 1)	1
  (6, 2)	1
  (7, 8)	1
  (8, 7)	1
  (9, 5)	1
  (10, 6)	1
  (11, 3)	1


In [ ]:
# Check shape
print(movie.shape)

# Missing values
print(movie.isnull().sum())

# Important columns
movie[['title', 'director', 'cast', 'listed_in', 'description']].head()

(8807, 12)
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64


,title,director,cast,listed_in,description
0,Dick Johnson Is Dead,Kirsten Johnson,NaN,Documentaries,"As her father nears the end of his life, filmm..."
1,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...","International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...","Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,Jailbirds New Orleans,NaN,NaN,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...","International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [ ]:
features = ['director', 'cast', 'listed_in', 'description']

for feature in features:
    movie[feature] = movie[feature].fillna('')

In [ ]:
movie['combined_features'] = (
    movie['director'] + ' ' +
    movie['cast'] + ' ' +
    movie['listed_in'] + ' ' +
    movie['description']
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
print(movie.columns.tolist())

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(movie['combined_features'])

print(tfidf_matrix.shape)
cosine_sim = cosine_similarity(tfidf_matrix)

print(cosine_sim.shape)

['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description', 'combined_features']
(8807, 5000)
(8807, 8807)


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

cv = CountVectorizer(stop_words='english')

matrix = cv.fit_transform(movie['combined_features'])

similarity = cosine_similarity(matrix)


def recommend(movie_name):

    idx = movie[movie['title'] == movie_name].index[0]

    scores = similarity[idx]

    similar_movies = sorted(
        list(enumerate(scores)),
        reverse=True,
        key=lambda x: x[1]
    )

    for i in similar_movies[1:6]:
        print(movie.iloc[i[0]]['title'])

recommend('The Conjuring')
recommend("Narcos")
recommend("The Crown")

The Conjuring 2
Insidious
In the Tall Grass
Creep
I Am the Pretty Thing That Lives in the House
Wild District
Narcos: Mexico
El final del paraíso
El Cartel
The Great Heist
My Hotter Half
How to Live Mortgage Free with Sarah Beeny
Dracula
Million Pound Menu
Back with the Ex
